# Covid-19, bakteriális és vírusos pneumonia elkülönítése mellkasröntgen felvételek alapján

*Neurális hálózat alapú klasszifikációs és szegmentációs modell építése*

*Ambrus Csaba*



## Pipeline attekintes

1. **Kornyezet es importok** - futtatasi kornyezet inicializalasa.
2. **Konfiguracio es kapcsolok** - globális kapcsolok (`RUN_FULL_RETRAIN`, stb.).
3. **Adatpipeline ellenorzes** - split konzisztencia validalasa.
4. **Modellezesi pipeline** - modellenkenti konfiguralt futasok.
5. **Eredmeny osszefuzes** - egységes `comparison_df` + leaderboard.
6. **Vizualizacio es explainability** - fo metrikak, gorbek, Grad-CAM/saliency.


## 0. Setup

In [ ]:
from pathlib import Path
import os

BRANCH = "main"
REPO_URL = "https://github.com/csambrus/CXR.git"
REPO_DIR = "/content/CXR"

if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")

%cd /content
if not os.path.exists(f"{REPO_DIR}/.git"):
    !git clone --branch {BRANCH} --single-branch {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch origin {BRANCH}
    !git checkout {BRANCH}
    !git pull origin {BRANCH}

%cd {REPO_DIR}
%pip install -r requirements.txt

!mkdir -p /root/.kaggle
!cp kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json

from src.runtime import setup_tensorflow_runtime

setup_tensorflow_runtime()


# 1. Adatok betöltése

In [ ]:
from pathlib import Path
from src.download_dataset import download_all_datasets

from src.config import (
    SEGMENTATION_RAW_DIR,
    SEGMENTATION_DATA_DIR,
)

print("SEGMENTATION_RAW_DIR:", SEGMENTATION_RAW_DIR)
print("SEGMENTATION_DATA_DIR:", SEGMENTATION_DATA_DIR)

download_all_datasets()


# 2. Szegmentáció

Ebben a blokkban fut a teljes szegmentacios pipeline:

- szegmentacios adatellenorzes,
- tanitas es kiertes,
- predikcios mintak,
- classifier variansok generalasa.


In [ ]:
from src.lung_segmentation import prepare_segmentation_dataset, create_splits, verify_png_files

verify_png_files(SEGMENTATION_RAW_DIR / "images", "Images")
verify_png_files(SEGMENTATION_RAW_DIR / "masks", "Masks")
prepare_segmentation_dataset(overwrite = False)
create_splits()

In [ ]:
from src.lung_segmentation import train_segmentation, evaluate_segmentation

train_segmentation(epochs=20)
evaluate_segmentation()

In [ ]:
from src.lung_segmentation import plot_training_history, plot_predictions

plot_training_history()
plot_predictions()

In [ ]:
from src.lung_segmentation import generate_classifier_variants

generate_classifier_variants()

# 3. Klasszifikáció

Ebben a blokkban fut a klasszifikacios pipeline a split ellenorzestol a vegso leaderboardig.


In [ ]:
import pandas as pd
from pathlib import Path

from src.config import (
    RAW_DIR,
    LUNG_MASKED_DIR,
    LUNG_CROP_DIR,
    MODELS_DIR,
    SPLITS_DIR,
    OUTPUT_DIR,
    ensure_dir,
)
from src.dataloader import print_split_summary
from src.compare_explainability import run_compare_explainability
from src.pipeline import (
    get_default_model_run_configs,
    run_training_pipeline,
    build_final_comparison,
    generate_final_plots,
    report_best_models,
    print_pipeline_leaderboard,
    save_pipeline_config,
)

RUN_FULL_RETRAIN = True

DATA_VARIANTS = ["raw", "lung_masked", "lung_crop"]
FINAL_COMPARISON_NAME = "project_final_optimized"
FINAL_OUT_DIR = ensure_dir(Path(MODELS_DIR) / FINAL_COMPARISON_NAME)

print("RAW_DIR        :", RAW_DIR)
print("LUNG_MASKED_DIR:", LUNG_MASKED_DIR)
print("LUNG_CROP_DIR  :", LUNG_CROP_DIR)
print("MODELS_DIR     :", MODELS_DIR)
print("SPLITS_DIR     :", SPLITS_DIR)
print("OUTPUT_DIR     :", OUTPUT_DIR)
print("FINAL_OUT_DIR  :", FINAL_OUT_DIR)


## 3.1 Split-ek képzése: train / valid / test: 70% / 15% / 15%

Itt ellenorizzuk, hogy a `raw`, `lung_masked` es `lung_crop` variansok ugyanazt a split logikat kovetik.

Ez biztositja, hogy a modell- es variansosszehasonlitas fair maradjon.

In [ ]:
from src.dataloader import create_splits, print_split_summary
from src.config import RAW_DIR, SPLITS_DIR
from pathlib import Path

# Convert RAW_DIR and SPLITS_DIR to Path objects if they are strings
source_root_path = Path(RAW_DIR)
split_dir_path = Path(SPLITS_DIR)

splits = create_splits(
    source_root=source_root_path,
    split_dir=split_dir_path,
    overwrite=True,
)

print_split_summary(split_dir_path)


## 3.2 Model Build Sanity Check

Gyors ellenorzes: az `efficientnetb0` modell pipeline-jaban szerepel-e a `scale_to_255` reteg.

Ha ez hianyzik, akkor a futo kod nem a vart training pipeline-t hasznalja.

In [ ]:
from src.train import build_model

_tmp_model, _tmp_base = build_model("efficientnetb0", pretrained=True)
layer_names = [layer.name for layer in _tmp_model.layers]
print("EfficientNet model layers:", layer_names)

if "scale_to_255" not in layer_names:
    raise RuntimeError(
        "[ERROR] EfficientNetB0 modelben nincs scale_to_255 réteg. "
        "Frissítsd / reloadold a javított train.py-t!"
    )
else:
    print("[OK] EfficientNetB0 scale_to_255 réteg megtalálva.")


## 3.3 Modellfutasi Konfiguraciok

Ebben a blokkban deklaraljuk a pipeline altal hasznalt futasi konfiguraciokat.

Minden modell sajat hiperparameter-beallitast kap, hogy a futas transzparens es reprodukalhato legyen.

In [ ]:
MODEL_RUN_CONFIGS = get_default_model_run_configs(DATA_VARIANTS)

save_pipeline_config(
    run_configs=MODEL_RUN_CONFIGS,
    final_out_dir=FINAL_OUT_DIR,
    run_full_retrain=RUN_FULL_RETRAIN,
    data_variants=DATA_VARIANTS,
)

pd.DataFrame([cfg.__dict__ for cfg in MODEL_RUN_CONFIGS])


## 3.4 Training Pipeline Futtatas

A training pipeline sorban futtatja az osszes konfiguraciot, es egy egyesitett eredmeny-DataFrame-et ad vissza.

`RUN_FULL_RETRAIN=False` eseten a mar kesz futasok ujrahasznalhatok.

In [ ]:
model_results_df = run_training_pipeline(
    split_dir=SPLITS_DIR,
    run_configs=MODEL_RUN_CONFIGS,
    run_full_retrain=RUN_FULL_RETRAIN,
    models_dir=MODELS_DIR,
)

print("[OK] Pipeline training lepes kesz.")
print("\n[CHECK] model x variant:")
print(model_results_df.groupby(["model", "data_variant"]).size())

print("\n[CHECK] models:")
print(model_results_df["model"].unique())

print("\n[CHECK] variants:")
print(model_results_df["data_variant"].unique())


## 3.5 Vegso Osszefuzes es Leaderboard

Itt epul fel a vegso `comparison_df`, deduplikalva `model + data_variant` kulcs alapjan.

A blokk egyszerre menti a leaderboard artefaktumokat is.

In [ ]:
comparison_df, FINAL_OUT_DIR = build_final_comparison(
    model_results_df=model_results_df,
    final_comparison_name=FINAL_COMPARISON_NAME,
    models_dir=MODELS_DIR,
)

print_pipeline_leaderboard(comparison_df)
display(comparison_df)


## 3.6 Vegso Vizualizacios Pipeline

A vegso `comparison_df` alapjan keszulnek a standard osszehasonlito abra-csomagok:

- osszesitett metric chartok,
- training gorbek,
- epoch-os osszehasonlitasok.

In [ ]:
generate_final_plots(comparison_df, FINAL_OUT_DIR, show=True)
print("[OK] Final comparison plots saved to:", FINAL_OUT_DIR)


## 3.7 Explainability Pipeline

Ebben a lepesben fut a Grad-CAM + saliency osszehasonlitas.

Celja, hogy vizualisan is ellenorizheto legyen, mit tanultak a modellek kulonbozo adatformakon.

In [ ]:
explainability_summary = run_compare_explainability(
    model_names=["resnet50", "vgg16", "efficientnetb0"],
    data_variants=DATA_VARIANTS,
    split_dir=SPLITS_DIR,
    out_dir=Path(OUTPUT_DIR) / "figures" / "project_final_explainability",
    n_examples=6,
    include_saliency=True,
    show=True,
    skip_existing=False,
)
explainability_summary


## 3.8 Best Model Kivalasztas

A blokk kivalasztja a legjobb modellt variansonkent es modellenkent a macro-F1 alapjan.

Ez a riportolas es deployment priorizalas alapja.

In [ ]:
best_by_variant, best_by_model = report_best_models(comparison_df, FINAL_OUT_DIR)

print("Best by variant:")
display(best_by_variant)

print("Best by model:")
display(best_by_model)


# 4. Zárás

Colab futas eseten a notebook mentesi kerest kuld, hogy a pipeline eredmenyei biztosan megmaradjanak.


In [ ]:
from pathlib import Path
import shutil

from src.config import IS_COLAB, OUTPUT_DIR, LOGS_DIR, GDRIVE_OUTPUT, ensure_dir

if IS_COLAB:
    gdrive_out = ensure_dir(GDRIVE_OUTPUT)
    output_src = Path(OUTPUT_DIR)
    logs_src = Path(LOGS_DIR)

    output_dst = gdrive_out / "outputs"
    logs_dst = gdrive_out / "logs"

    if output_src.exists():
        shutil.copytree(output_src, output_dst, dirs_exist_ok=True)
        print("[OK] OUTPUT_DIR copied to:", output_dst)
    else:
        print("[WARN] OUTPUT_DIR does not exist:", output_src)

    if logs_src.exists():
        shutil.copytree(logs_src, logs_dst, dirs_exist_ok=True)
        print("[OK] LOGS_DIR copied to:", logs_dst)
    else:
        print("[WARN] LOGS_DIR does not exist:", logs_src)
else:
    print("[INFO] Not running in Colab, skip Drive sync.")

try:
    from google.colab import _message
    _message.blocking_request("request_save", timeout_sec=10)
    print("Notebook save requested.")
except Exception as e:
    print("[WARN] Notebook save request failed or not running in Colab:", e)
